# Public-release note

This curated notebook contains the original research workflow with execution outputs removed. Bloomberg and other licensed source data are not distributed. To run it, supply compatible files under `data/processed/` as described in `data/README.md`. Any results produced locally depend on the user's licensed data and are not included in this repository.


# Bond Total-Return Construction and Validation

This notebook validates synthetic government-bond total-return series constructed from 10-year yields against an external benchmark. It demonstrates the duration-based return approximation implemented in `src/data/bond_returns.py`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Locate the repository root when running from either the project or notebooks directory
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.bond_returns import build_bond_total_return_index, validate_against_benchmark #importo funzioni create in bond_returns.py

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
TABLES_DIR = PROJECT_ROOT / "results" / "tables"
PLOTS_DIR = PROJECT_ROOT / "plots" / "bond_validation"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load cleaned intermediate datasets
yields = pd.read_csv(INTERIM_DIR / "cleaned_yields.csv", parse_dates=["Date"])
benchmarks = pd.read_csv(INTERIM_DIR / "cleaned_10y_bond_tr_benchmarks.csv", parse_dates=["Date"])

yields.head(), benchmarks.head()

In [ ]:
# Construct synthetic 10Y total returns using country-specific coupon frequencies
synthetic = {  # one synthetic return data frame per country
    "US": build_bond_total_return_index(
        yields,
        "US_10Y_Yield",  #prendo colonna US_10y_Yield
        maturity=10,  #fisso maturity (float)
        coupon_freq=2,  #fisso frequenza cedolare
        index_name="US_10Y_Synthetic",
    ),
    "UK": build_bond_total_return_index(
        yields,
        "UK_10Y_Yield_BOE",
        maturity=10,
        coupon_freq=1,
        index_name="UK_10Y_Synthetic",
    ),
    "DE": build_bond_total_return_index(
        yields,
        "GER_10Y_Yield_BBK",
        maturity=10,
        coupon_freq=1,
        index_name="DE_10Y_Synthetic",
    ),
}

{country: df[["Date"]].agg(["min", "max"]).to_dict()["Date"] for country, df in synthetic.items()}  # coverage of the three synthetic indices

In [ ]:
# Compare synthetic and benchmark returns over the common sample
configs = {  # synthetic and benchmark return columns by country
    "US": ("US_10Y_Synthetic_Return", "US_10Y_Benchmark_Return"),
    "UK": ("UK_10Y_Synthetic_Return", "UK_10Y_Benchmark_Return"),
    "DE": ("DE_10Y_Synthetic_Return", "DE_10Y_Benchmark_Return"),
}

comparisons, stats = {}, []
# Store comparison data frames and validation statistics

for country, (syn_col, bench_col) in configs.items():
    comp, stat = validate_against_benchmark(
        synthetic[country],
        benchmarks,
        synthetic_return_col=syn_col,
        benchmark_return_col=bench_col,
    )
    comp["Country"] = country
    comparisons[country] = comp
    stats.append({"Country": country, **stat})

stats_df = pd.DataFrame(stats)
stats_df

In [ ]:
# Save detailed statistics and one Excel sheet per country
comparison_df = pd.concat(comparisons.values(), ignore_index=True)

stats_df.to_csv(TABLES_DIR / "bond_10y_validation_stats.csv", index=False)
comparison_df.to_csv(TABLES_DIR / "bond_10y_validation_comparison.csv", index=False)

xlsx_path = TABLES_DIR / "bond_10y_validation.xlsx"

# If the workbook is locked, save an alternative copy
def write_validation_excel(path):
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        stats_df.to_excel(writer, sheet_name="stats", index=False)
        for country, comp in comparisons.items():
            cols = ["Date", configs[country][0], configs[country][1], "Return_Diff"]
            comp[cols].to_excel(writer, sheet_name=f"{country}_monthly", index=False)

        # Apply readable date, percentage, and column-width formatting
        for ws in writer.book.worksheets:
            ws.freeze_panes = "A2"
            for cell in ws[1]:
                cell.style = "Headline 4"
            for col in ws.columns:
                letter = col[0].column_letter
                width = max(len(str(cell.value)) if cell.value is not None else 0 for cell in col[:200]) + 2
                ws.column_dimensions[letter].width = min(max(width, 12), 28)
            for row in ws.iter_rows(min_row=2):
                for cell in row:
                    if cell.column == 1 and ws.title != "stats":
                        cell.number_format = "yyyy-mm-dd"
                    elif ws.title == "stats" and cell.column == 3:
                        cell.number_format = "0.0000"
                    elif isinstance(cell.value, float):
                        cell.number_format = "0.00%"
            if ws.title != "stats":
                ws.column_dimensions["A"].width = 12

try:
    write_validation_excel(xlsx_path)
except PermissionError:
    xlsx_path = TABLES_DIR / "bond_10y_validation_copy.xlsx"
    write_validation_excel(xlsx_path)

xlsx_path


In [ ]:
# Rebuild cumulative indices over the common sample and rebase to 100
def cumulative_from_returns(ret: pd.Series, base: float = 100) -> pd.Series:
    return base * (1 + ret.fillna(0)).cumprod()

plot_data = {}

for country, (syn_col, bench_col) in configs.items():
    comp = comparisons[country].copy()
    comp[f"{country}_Synthetic_Index"] = cumulative_from_returns(comp[syn_col])
    comp[f"{country}_Benchmark_Index"] = cumulative_from_returns(comp[bench_col])
    plot_data[country] = comp

plot_data["US"].head()

In [ ]:
# Grafici cumulati
import matplotlib.pyplot as plt

# White background for print-ready charts
plt.style.use("default")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "axes.titlecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
})

for country, comp in plot_data.items():
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(comp["Date"], comp[f"{country}_Synthetic_Index"], label="Synthetic 10Y")
    ax.plot(comp["Date"], comp[f"{country}_Benchmark_Index"], label="Benchmark 10Y", linestyle="--")
    ax.set_title(f"{country} 10Y bond total return: sintetico vs benchmark")
    ax.set_ylabel("Cumulative index, common sample = 100")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / f"{country.lower()}_10y_bond_validation.png", dpi=150, facecolor="white")
    plt.show()

## Interpreting the validation statistics

- `corr` measures co-movement between synthetic returns and the benchmark.
- `mae` and `rmse` measure average approximation error.
- `mean_synthetic` and `mean_benchmark` compare average monthly returns.
- `vol_synthetic` and `vol_benchmark` compare monthly volatility.

The validation is intended to assess whether the synthetic series is a reasonable research proxy; it does not imply that the benchmark data may be redistributed.